# Reduzierung der Sensorredundanz in der Fertigungslinie mit PROC GVARCLUS


## Zusammenfassung

Eine mehrzonige Fertigungslinie liefert fortlaufend Dutzende korrelierter Sensormesswerte, von denen viele dasselbe zugrunde liegende Signal tragen. Dieses Notebook verwendet **PROC GVARCLUS** (Variablenclusterung), um die 13 Prozesssensoren in vier disjunkte Cluster zu gruppieren, liest anschließend die R-Quadrat-Werte jedes Clusters aus, um pro Cluster einen repräsentativen Messfühler auszuwählen — und verkleinert so den SPC-Überwachungsaufwand, ohne Prozessinformation zu verlieren. Drei der vier Cluster lassen sich sauber physischen Zonen zuordnen (durchschnittliches R-Quadrat 0,92, 0,93 und 0,96); der vierte ist ein Einzelkanal-Cluster, den die Prozedur abgespalten hat und den wir genauer untersuchen, statt ihn zu übergehen.


## Datenquellen

Alle Daten werden inline mit `call streaminit(20260531)` und `rand()` erzeugt — keine externen oder Netzwerkeingaben.

| Datensatz | Zeilen | Schlüsselvariablen | Beschreibung |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Synthetische Messwerte einer 3-Zonen-Fertigungslinie. Sensoren innerhalb einer Zone teilen sich einen latenten Treiber (hohe Korrelation); zonenübergreifende Messfühler (Temperaturen an Zone 1 gekoppelt, Drücke an Zone 2, Vibration an Zone 3) werden hinzugefügt, um realistische Redundanz zu erzeugen. Die DATA-Schritt-Schleife fordert 300 Zyklen an, aber diese Testumgebung läuft im unlizenzierten Modus und behält nur die ersten 100 Beobachtungen — genug, um die Clusterstruktur sauber zu rekonstruieren. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Eine Zeile je Eingabesensor: der zugewiesene Cluster und sein R-Quadrat mit der Komponente dieses Clusters. Erzeugt durch die `OUTPUT OUT=`-Anweisung. |


# Reduzierung der Sensorredundanz in der Fertigungslinie mit PROC GVARCLUS

Auf einer instrumentierten Fertigungslinie ist es üblich, Dutzende Sensoren zu erfassen, die sich überlappende physikalische Phänomene messen — mehrere Thermoelemente in einer Zone, redundante Drucksensoren, Vibrationssonden, die dieselbe Welle überwachen. Jeden Kanal in eine Regelkarte oder ein nachgelagertes Modell einzuspeisen, verschwendet Überwachungsaufwand und verstärkt die Multikollinearität.

**Variablenclusterung** löst dieses Problem direkt. `PROC GVARCLUS` teilt die numerischen Variablen in *disjunkte* Cluster auf, wobei jeder Cluster durch die erste Hauptkomponente seiner Mitglieder zusammengefasst wird. Sensoren, die sich gemeinsam bewegen, landen im selben Cluster; der Ingenieur kann dann einen repräsentativen Kanal pro Cluster beibehalten.

In diesem Notebook werden wir:

1. Messwerte einer 3-Zonen-Linie mit absichtlich redundanten Sensoren synthetisieren (die Schleife fordert 300 Zyklen an; hier werden 100 beibehalten).
2. Die 13 Kanäle mit `PROC GVARCLUS` bei `MAXCLUSTERS=4` clustern.
3. Die Clusterzuweisungen in einem Ausgabedatensatz erfassen und zusammenfassen.
4. Die R-Quadrat-Werte interpretieren, um pro Cluster einen Messfühler für die laufende SPC auszuwählen.


## Schritt 1 — Synthetische Sensordaten erzeugen

Wir simulieren drei Prozesszonen, jede mit einem verborgenen Treiber (`zone1_a`, `zone2_a`, `zone3_a`). Die übrigen Kanäle sind verrauschte lineare Funktionen des Treibers ihrer Zone, sodass sie innerhalb einer Zone stark korreliert, zonenübergreifend jedoch nahezu unabhängig sind. Außerdem koppeln wir die Ein-/Auslasstemperaturen an Zone 1 und die beiden Drucksensoren an Zone 2, um die geräteübergreifende Redundanz realer Linien nachzubilden. Ein fester Seed macht den Lauf reproduzierbar. Die Schleife fordert 300 Zyklen an; im unlizenzierten Modus behält die Engine die ersten 100 Beobachtungen, was die untenstehende NOTE meldet.


In [1]:
DATEN process_sensors;
    AUFRUFEN streaminit(20260531);
    AUSFÜHRUNG cycle = 1 BIS 300;
        /* Zone 1: latenter Treiber plus zwei redundante Sonden */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latenter Treiber plus zwei redundante Sonden */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latenter Treiber plus eine redundante Sonde */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Geräteübergreifende Kanäle, gekoppelt an bestehende Treiber */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        AUSGABE;
    ENDE;
    ENTFERNEN cycle;
AUSFÜHREN;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Schritt 2 — Die Sensoren clustern

Wir rufen `PROC GVARCLUS` für alle 13 Kanäle auf. Die `VAR`-Anweisung listet die zu clusternden numerischen Sensoren auf. Wir begrenzen die Partition mit `MAXCLUSTERS=4` auf vier Cluster (wir erwarten etwa einen Cluster pro physischer Zone, mit etwas Spielraum). `ODS GRAPHICS ON` mit `PLOTS=ALL` aktiviert das Variablen-Cluster-Dendrogramm; die Engine schreibt dieses SVG in das Arbeitsverzeichnis, anstatt es inline zu rendern, sodass die Struktur, die wir unten ablesen, aus der gedruckten Tabelle **Variable Cluster Assignments** und der Eigenwerttabelle je Cluster stammt.

Die `OUTPUT OUT=`-Anweisung schreibt die Variablen-zu-Cluster-Zuweisungen — zusammen mit dem R-Quadrat jeder Variablen gegenüber ihrem eigenen Cluster — in einen Datensatz, mit dem wir später weiterarbeiten können.


In [2]:
ODS GRAPHICS ON;

PROZEDUR gvarclus DATEN=process_sensors maxclusters=4 PLOTS=ALL;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    AUSGABE out=clusters;
AUSFÜHREN;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Schritt 3 — Erneuter Lauf mit der Option SUMMARY

Ein zweiter Lauf der Prozedur mit der Option `SUMMARY` behält dieselbe Partition bei. `PLOTS=NONE` schaltet in diesem Durchgang die Grafikausgabe ab. In dieser Umgebung ist der gedruckte Bericht identisch mit Schritt 2 — dieselbe 13-zeilige Zuweisungstabelle, dieselben vier Cluster und dieselbe Eigenwert-Zusammenfassung — daher zeigt diese Zelle vor allem, dass die Optionen `SUMMARY` und `PLOTS=NONE` geparst werden und laufen; es werden keine neuen Zahlen erwartet.


In [3]:
PROZEDUR gvarclus DATEN=process_sensors maxclusters=4 summary PLOTS=none;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
AUSFÜHREN;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Schritt 4 — Den Ausgabedatensatz untersuchen

Der Datensatz `clusters` aus `OUTPUT OUT=` enthält eine Zeile pro Sensor mit seinem zugewiesenen `Cluster` und `RSq_Own` (der quadrierten Korrelation zwischen dem Sensor und der Komponente seines Clusters). Ein hoher `RSq_Own`-Wert bedeutet, dass der Sensor gut durch seinen Cluster repräsentiert wird — ein idealer Kandidat, um ihn zugunsten des Cluster-Repräsentanten *fallenzulassen*. Wir sortieren nach Cluster und dann nach R-Quadrat, sodass der repräsentativste Sensor jedes Clusters oben in seiner Gruppe steht.


In [4]:
PROZEDUR SORTIEREN DATEN=clusters out=clusters_ranked;
    NACH CLUSTER ABSTEIGEND RSq_Own;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=clusters_ranked BEZEICHNUNG;
    VAR Variable CLUSTER RSq_Own;
    BEZEICHNUNG Variable = "Sensorkanal"
          CLUSTER  = "Cluster"
          RSq_Own  = "R-Quadrat zum eigenen Cluster";
AUSFÜHREN;


  Obs  Sensorkanal  Cluster  R-Quadrat zum eigenen Cluster
-----  -----------  -------  -----------------------------
    1  ZONE1_A            1                        0.96867
    2  ZONE1_B            1                         0.9189
    3  TEMP_IN            1                       0.903394
    4  TEMP_OUT           1                       0.889519
    5  ZONE2_A            2                        0.98364
    6  ZONE2_B            2                       0.946708
    7  PRESSURE_A         2                       0.929356
    8  PRESSURE_B         2                       0.920915
    9  ZONE2_C            2                       0.892405
   10  ZONE3_A            3                       0.977237
   11  VIBRATION          3                        0.95916
   12  ZONE3_B            3                       0.949054
   13  ZONE1_C            4                              1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpretation der Ergebnisse

Die Partition rekonstruiert den größten Teil der physischen Struktur der Linie, mit einem ehrlichen Vorbehalt:

- **Cluster 1** vereint `zone1_a` (R²=0,969), `zone1_b` (0,919) sowie die Ein-/Auslasstemperaturen `temp_in` (0,903) und `temp_out` (0,890) — vier der fünf Kanäle, die vom latenten Signal der Zone 1 angetrieben werden. Durchschnittliches R-Quadrat 0,920.
- **Cluster 2** vereint alle drei Zone-2-Kanäle `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) zusammen mit den beiden Drucksensoren `pressure_a` (0,929) und `pressure_b` (0,921), die wir an den Zone-2-Treiber gekoppelt haben. Durchschnittliches R-Quadrat 0,935.
- **Cluster 3** vereint `zone3_a` (0,977), `zone3_b` (0,949) und `vibration` (0,959). Durchschnittliches R-Quadrat 0,962.
- **Cluster 4** besteht aus einem einzigen Kanal: `zone1_c`, die dritte Zone-1-Sonde, wurde mit R²=1,000 als eigener Cluster abgespalten (ein Einzelcluster wird trivialerweise perfekt durch sich selbst erklärt). Obwohl `zone1_c` denselben Zone-1-Treiber teilt, beurteilte die Prozedur bei dieser Stichprobengröße `zone1_c` als hinreichend verschieden von der `zone1_a`/Temperatur-Gruppe, um einen eigenen Cluster zu rechtfertigen, statt ihn in Cluster 1 einzugliedern.

Die drei Mehrkanal-Cluster weisen jeweils ein durchschnittliches R-Quadrat über **0,90** auf, was bestätigt, dass eine einzige Komponente den Großteil der Variation unter ihren Mitgliedern erklärt — genau die Redundanz, die wir zusammenfassen möchten. Der einzelne Ausreißer — `zone1_c`, der in seinem eigenen Cluster landet statt beim Rest von Zone 1 — erinnert nützlich daran, dass Variablenclusterung datengetrieben ist: Mit mehr Zyklen oder einem höheren `MAXCLUSTERS`-Budget kann sich die Grenze verschieben, sodass die Partition ein Ausgangspunkt für technisches Urteilsvermögen ist, keine feste Wahrheit.

**Maßnahme für die Linie.** Behalten Sie innerhalb jedes Mehrkanal-Clusters den Sensor mit dem höchsten `RSq_Own`-Wert (den Kanal, der seinen Cluster am besten repräsentiert) und ziehen Sie die übrigen aus der routinemäßigen SPC-Erfassung zurück oder priorisieren Sie sie niedriger — zum Beispiel `zone2_a` für Cluster 2 und `zone3_a` für Cluster 3. Behandeln Sie den Einzelcluster `zone1_c` als Prüfhinweis statt als automatischen Verbleib: Klären Sie, ob er tatsächlich eigenständige Information trägt oder ein Artefakt der Clusterung ist, bevor Sie entscheiden, ihn separat zu überwachen. Selbst wenn dieser eine Kanal gesondert betrachtet wird, verdichtet dies 13 überwachte Kanäle zu einem Vier-Kanal-Überwachungsplan, wodurch Alarmrauschen und Multikollinearität reduziert werden, während der Informationsgehalt der Linie erhalten bleibt.
